In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get("https://www.google.com")
    print(f"✅ CONECTADO! Título de la página: {driver.title}")
    driver.quit()

except Exception as e:
    print(f"❌ Error: {e}")

✅ CONECTADO! Título de la página: Google


In [10]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO
# ==========================================================

URL_OBJETIVO = "https://www.coordinador.cl/"

SELECTOR_CONTENEDOR = "article"
SELECTOR_DATO_A = "h2, h3"
SELECTOR_DATO_B = "time, span"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print("🔎 Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5)

    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0
            })
        except:
            continue

    # ==========================================================
    # 4. SALIDA DE DATOS
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f"✅ Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())
    else:
        print("⚠️ No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print("🛑 Navegador cerrado.")

🔎 Conectando a la fuente de datos...
Se encontraron 17 registros potenciales.
✅ Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
                                     variable_nombre variable_valor  status
0  Coordinador Eléctrico realiza llamado a licita...         302026     1.0
1  Coordinador Eléctrico realiza Jornada Técnica ...         252026     1.0
2  Coordinador Eléctrico realiza llamado a licita...         192026     1.0
3  Ministra de Energía visitó el Centro de Despac...         162026     1.0
4  Coordinador Eléctrico informa mantenimiento de...         132026     1.0
🛑 Navegador cerrado.


In [25]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

datos_totales = []
paginas_objetivo = 5

try:
    driver.get("https://quotes.toscrape.com/")

    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
        )

        productos = driver.find_elements(By.CLASS_NAME, "quote")

        for p in productos:
            datos_totales.append(p.text)

        try:
            boton_siguiente = driver.find_element(
                By.XPATH, "//li[@class='next']/a"
            )

            boton_siguiente.click()
            time.sleep(2)

        except Exception:
            print("Se ha llegado a la última página.")
            break

    print("\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    driver.quit()

--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
--- Procesando Página 4 ---
--- Procesando Página 5 ---

Proceso finalizado con éxito.
Total de registros capturados: 50


In [27]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# CONFIGURACIÓN
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

datos_totales = []
paginas_objetivo = 5

try:
    driver.get("https://quotes.toscrape.com/")

    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        # ESPERA EXPLÍCITA (mejor que time.sleep)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
        )

        productos = driver.find_elements(By.CLASS_NAME, "quote")

        for p in productos:
            datos_totales.append(p.text)

        try:
            # XPATH del botón siguiente
            boton_siguiente = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//li[@class='next']/a"))
            )

            boton_siguiente.click()

        except Exception:
            print("Se ha llegado a la última página.")
            break

    print("\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    driver.quit()

--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
--- Procesando Página 4 ---
--- Procesando Página 5 ---

Proceso finalizado con éxito.
Total de registros capturados: 50
